# HW14: FAISS + sentence-transformers + mini-RAG

Учебная база знаний в коде.

In [ ]:
import os, re, json, random
from pathlib import Path

# Токен: https://huggingface.co/settings/tokens
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import faiss
from sentence_transformers import SentenceTransformer
from transformers import logging as tr_logging

tr_logging.set_verbosity_error()

ROOT = Path.cwd().resolve()
if ROOT.name != "HW14":
    cand = ROOT / "homeworks" / "HW14"
    if cand.is_dir():
        os.chdir(cand)
ART = Path("artifacts")
ART.mkdir(exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# small synthetic knowledge base (10-30 docs)
docs = [
    {"id": "d1", "text": "Курс AIE: модуль PyTorch вводит тензоры, autograd и nn.Module."},
    {"id": "d2", "text": "Регуляризация: dropout и weight decay уменьшают переобучение."},
    {"id": "d3", "text": "Оптимизатор Adam адаптивно масштабирует шаги по параметрам."},
    {"id": "d4", "text": "Сверточные сети используют локальные фильтры на изображениях."},
    {"id": "d5", "text": "ResNet использует остаточные связи для обучения глубоких сетей."},
    {"id": "d6", "text": "FAISS строит индекс для быстрого поиска ближайших векторов."},
    {"id": "d7", "text": "Mini-RAG: retrieval контекста затем генерация ответа."},
    {"id": "d8", "text": "Временные ряды требуют split по времени, не случайного перемешивания."},
    {"id": "d9", "text": "BERT использует self-attention для контекстных эмбеддингов токенов."},
    {"id": "d10", "text": "Оценка retrieval: hit@k и recall@k по заранее заданным релевантным документам."},
]

def chunk_text(text, size=80, overlap=20):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        piece = " ".join(words[i : i + size])
        chunks.append(piece)
        i += max(1, size - overlap)
    return chunks

# sanity-check базы и пример чанкинга (S14)
print("документов в базе:", len(docs))
for d in docs[:5]:
    print(d["id"], ":", d["text"][:100])
ex_chunks = chunk_text(docs[0]["text"])
print(
    "пример чанкинга:",
    docs[0]["id"],
    "-> чанков:",
    len(ex_chunks),
    "| первый:",
    ex_chunks[0][:90],
)

records = []
for d in docs:
    for j, ch in enumerate(chunk_text(d["text"])):
        records.append({"source": d["id"], "chunk_id": f"{d['id']}_c{j}", "text": ch})

model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [r["text"] for r in records]
emb = model.encode(texts, normalize_embeddings=True, show_progress_bar=False).astype("float32")
dim = emb.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(emb)

queries = [
    ("Что такое dropout?", "d2"),
    ("Как ускорить поиск по векторам?", "d6"),
    ("Что такое ResNet?", "d5"),
    ("Как валидировать временной ряд?", "d8"),
    ("Что делает BERT?", "d9"),
    ("Как оценить retrieval?", "d10"),
    ("Что такое PyTorch nn?", "d1"),
    ("Что такое Adam?", "d3"),
]

def search(q, k=3):
    qv = model.encode([q], normalize_embeddings=True, show_progress_bar=False).astype("float32")
    scores, idxs = index.search(qv, k)
    return [(records[i]["source"], records[i]["text"][:80], float(scores[0][j])) for j, i in enumerate(idxs[0])]

k_eval = 3
hits = []
rows_eval = []
for q, exp in queries:
    res = search(q, k=k_eval)
    retrieved_sources = [r[0] for r in res]
    hit = int(exp in retrieved_sources)
    hits.append(hit)
    rows_eval.append({"query": q, "expected_source": exp, "retrieved_sources": ";".join(retrieved_sources), "hit_at_k": hit})

hit_rate = np.mean(hits)
recall_k = hit_rate  # one relevant doc per query
print("hit@k", hit_rate, "recall@k", recall_k)

pd.DataFrame(rows_eval).to_csv(ART / "retrieval_eval.csv", index=False)

# experiment: chunk_size 40 vs 80
def build_index_chunk_size(cs):
    recs = []
    for d in docs:
        for j, ch in enumerate(chunk_text(d["text"], size=cs, overlap=10)):
            recs.append({"source": d["id"], "text": ch})
    e = model.encode([r["text"] for r in recs], normalize_embeddings=True, show_progress_bar=False).astype("float32")
    ix = faiss.IndexFlatIP(e.shape[1])
    ix.add(e)
    return recs, ix

r40, i40 = build_index_chunk_size(40)
r80, i80 = build_index_chunk_size(80)

def eval_ix(recs, ix):
    hs = []
    for q, exp in queries:
        qv = model.encode([q], normalize_embeddings=True, show_progress_bar=False).astype("float32")
        _, idxs = ix.search(qv, 3)
        srcs = [recs[i]["source"] for i in idxs[0]]
        hs.append(int(exp in srcs))
    return np.mean(hs)

print("chunk40", eval_ix(r40, i40), "chunk80", eval_ix(r80, i80))

# update KB
docs2 = docs + [
    {"id": "d11", "text": "Новый документ: FAISS поддерживает IVF и HNSW для больших баз."},
    {"id": "d12", "text": "Обновление политики: всегда логировать источники в RAG."},
]
records2 = []
for d in docs2:
    for j, ch in enumerate(chunk_text(d["text"])):
        records2.append({"source": d["id"], "text": ch})
emb2 = model.encode([r["text"] for r in records2], normalize_embeddings=True, show_progress_bar=False).astype("float32")
index2 = faiss.IndexFlatIP(emb2.shape[1])
index2.add(emb2)

q_spec = "Что такое IVF в FAISS?"
before = search(q_spec)  # old index still about general FAISS
qv = model.encode([q_spec], normalize_embeddings=True, show_progress_bar=False).astype("float32")
_, idxs = index2.search(qv, 3)
after = [(records2[i]["source"], records2[i]["text"][:80]) for i in idxs[0]]

pd.DataFrame([{
    "query": q_spec,
    "before_retrieved_sources": ";".join([x[0] for x in search(q_spec)]),
    "after_retrieved_sources": ";".join([a[0] for a in after]),
    "changed": str([x[0] for x in search(q_spec)] != [a[0] for a in after]),
}]).to_csv(ART / "retrieval_before_after_update.csv", index=False)

# mini-RAG
def rag_answer(question):
    ctx = search(question, k=2)
    context = " ".join([c[1] for c in ctx])
    answer = "Кратко: " + context[:200].replace("\n", " ") + "."
    return answer, [c[0] for c in ctx]

rag_rows = []
for q, _ in queries[:5]:
    ans, src = rag_answer(q)
    rag_rows.append({"question": q, "answer": ans, "retrieved_sources": ";".join(src)})
pd.DataFrame(rag_rows).to_csv(ART / "rag_examples.csv", index=False)
print("HW14 done")